# Weather- and Lighting-Robust Traffic Sign Classification
### Multi-Task Learning with a Synthetic Adverse-Condition Dataset

This notebook trains and evaluates three models on GTSRB:

| Model | Training Data | Output Heads |
|-------|---------------|--------------|
| **Baseline**  | Clean images only        | Sign class (43) |
| **Augmented** | Clean + 7 corruptions    | Sign class (43) |
| **Multi-Task**| Clean + 7 corruptions    | Sign class (43) **+** Condition (8) |

The three-way comparison isolates two effects:
- **Baseline vs Augmented** — effect of training on synthetic adverse data.
- **Augmented vs Multi-Task** — effect of the auxiliary condition-prediction head.

Outputs: trained model checkpoints, per-condition accuracy table, performance-drop chart,
confusion matrices, qualitative success/failure examples, ablation on corruption probability,
and a real-world evaluation.

## 1. Setup

In [ ]:
!pip install -q albumentations opencv-python-headless tqdm


In [ ]:
# Standard library
import os, json, random, shutil, copy, math
from pathlib import Path
from collections import defaultdict

# Numerical / image
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Progress bar
from tqdm import tqdm

# PIL
from PIL import Image

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# torchvision
from torchvision import transforms, models
from torchvision.datasets import GTSRB

# scikit-learn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Project directory layout
PROJECT_DIR = Path("/content/traffic_sign_robustness_project")
MODELS_DIR  = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR    = Path("/content/data")          # GTSRB download target

for p in [PROJECT_DIR, MODELS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)


## 2. Label dictionaries

GTSRB has 43 sign classes. Our synthetic dataset adds an 8-class **condition** label:
0 = clean, 1–7 = the seven corruption types.

In [ ]:
class_names = {
     0: "Speed limit (20km/h)",      1: "Speed limit (30km/h)",
     2: "Speed limit (50km/h)",      3: "Speed limit (60km/h)",
     4: "Speed limit (70km/h)",      5: "Speed limit (80km/h)",
     6: "End of speed limit (80km/h)", 7: "Speed limit (100km/h)",
     8: "Speed limit (120km/h)",     9: "No passing",
    10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection", 12: "Priority road",
    13: "Yield", 14: "Stop", 15: "No vehicles",
    16: "Vehicles over 3.5 metric tons prohibited", 17: "No entry",
    18: "General caution",          19: "Dangerous curve to the left",
    20: "Dangerous curve to the right", 21: "Double curve",
    22: "Bumpy road",               23: "Slippery road",
    24: "Road narrows on the right", 25: "Road work",
    26: "Traffic signals",          27: "Pedestrians",
    28: "Children crossing",        29: "Bicycles crossing",
    30: "Beware of ice/snow",       31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",         34: "Turn left ahead",
    35: "Ahead only",               36: "Go straight or right",
    37: "Go straight or left",      38: "Keep right",
    39: "Keep left",                40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons",
}

condition_names = {
    0: "clean",
    1: "low_brightness",
    2: "overexposure",
    3: "shadow",
    4: "fog",
    5: "rain",
    6: "motion_blur",
    7: "low_contrast",
}

with open(PROJECT_DIR / "class_names.json", "w") as f:
    json.dump(class_names, f, indent=4)
with open(PROJECT_DIR / "condition_names.json", "w") as f:
    json.dump(condition_names, f, indent=4)

NUM_CLASSES    = len(class_names)        # 43
NUM_CONDITIONS = len(condition_names)    # 8
print(f"Classes: {NUM_CLASSES}, Conditions: {NUM_CONDITIONS}")


## 3. Dataset and configuration

We load GTSRB through `torchvision`, then take a fixed random subset for compute-efficient
training. Subsetting is reproducible (`SEED = 42`) and the same train/test indices are used
across all three models so comparisons are fair.

In [ ]:
train_dataset_raw = GTSRB(root=str(DATA_DIR), split="train", download=True)
test_dataset_raw  = GTSRB(root=str(DATA_DIR), split="test",  download=True)

print("Full train size:", len(train_dataset_raw))
print("Full test size:",  len(test_dataset_raw))


In [ ]:
# ---------------------- HYPERPARAMETERS ----------------------
IMAGE_SIZE        = 64           # GTSRB signs are small; 64x64 is standard for this task
BATCH_SIZE        = 128
TRAIN_SUBSET_SIZE = 20000        # of 26640 — large enough for stable results, small enough to be fast
TEST_SUBSET_SIZE  = 5000         # of 12630
EPOCHS            = 6
LEARNING_RATE     = 1e-4
CORRUPTION_PROB   = 0.7          # P(apply a random corruption | training sample)
AUX_LOSS_WEIGHT   = 0.3          # multi-task: total_loss = sign_loss + AUX_LOSS_WEIGHT * cond_loss

# Reproducible subset indices — SAME across all three models for fair comparison
rng = random.Random(SEED)

train_indices = list(range(len(train_dataset_raw)))
rng.shuffle(train_indices)
train_indices = train_indices[:TRAIN_SUBSET_SIZE]

test_indices = list(range(len(test_dataset_raw)))
rng.shuffle(test_indices)
test_indices = test_indices[:TEST_SUBSET_SIZE]

print(f"Training subset:  {len(train_indices)}")
print(f"Test subset:      {len(test_indices)}")


In [ ]:
# ImageNet normalization, since the backbone is ImageNet-pretrained
clean_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


## 4. Synthetic corruption pipeline

Seven OpenCV-based degradations, each parameterized to roughly match the visual severity of
real-world adverse conditions. Each function takes a PIL image and returns a PIL image.

In [ ]:
def pil_to_cv2(pil_img):
    return np.array(pil_img.convert("RGB"))

def cv2_to_pil(arr):
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

# ---- 1. Low brightness (underexposure / dusk) ----
def apply_low_brightness(img):
    return cv2.convertScaleAbs(img, alpha=0.55, beta=-20)

# ---- 2. Overexposure (intense direct sunlight) ----
def apply_overexposure(img):
    return cv2.convertScaleAbs(img, alpha=1.45, beta=45)

# ---- 3. Shadow overlay ----
def apply_shadow(img):
    h, w, _ = img.shape
    shadow = img.copy()
    x1 = random.randint(0, w // 2)
    x2 = random.randint(w // 2, w)
    polygon = np.array([[(x1, 0), (x2, 0), (x2, h), (x1, h)]], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, polygon, 255)
    shadow_factor = 0.45
    shadow[mask == 255] = (shadow[mask == 255] * shadow_factor).astype(np.uint8)
    return shadow

# ---- 4. Fog (white haze + contrast loss) ----
def apply_fog(img):
    fog_layer = np.full(img.shape, 200, dtype=np.uint8)
    return cv2.addWeighted(img, 0.55, fog_layer, 0.45, 0)

# ---- 5. Rain streaks ----
def apply_rain(img):
    rain_img = img.copy()
    h, w, _  = rain_img.shape
    n_drops  = 600
    for _ in range(n_drops):
        x = random.randint(0, w - 1)
        y = random.randint(0, h - 1)
        length = random.randint(8, 14)
        cv2.line(rain_img, (x, y), (x - 1, y + length), (220, 220, 220), 1)
    rain_img = cv2.blur(rain_img, (3, 3))
    rain_img = cv2.convertScaleAbs(rain_img, alpha=0.85, beta=-10)
    return rain_img

# ---- 6. Motion blur (horizontal kernel) ----
def apply_motion_blur(img):
    k = 9
    kernel = np.zeros((k, k))
    kernel[k // 2, :] = 1.0 / k
    return cv2.filter2D(img, -1, kernel)

# ---- 7. Global low contrast ----
def apply_low_contrast(img):
    return cv2.convertScaleAbs(img, alpha=0.55, beta=40)


CONDITION_FUNCS = {
    1: apply_low_brightness,
    2: apply_overexposure,
    3: apply_shadow,
    4: apply_fog,
    5: apply_rain,
    6: apply_motion_blur,
    7: apply_low_contrast,
}

def apply_condition(pil_img, condition_id):
    """Apply a corruption by its condition_id (0 = clean, 1–7 = corruptions)."""
    if condition_id == 0:
        return pil_img
    arr = pil_to_cv2(pil_img)
    arr = CONDITION_FUNCS[condition_id](arr)
    return cv2_to_pil(arr)


In [ ]:
# Visual sanity check: one sign through all 8 conditions
sample_img, sample_label = train_dataset_raw[10]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for cid in range(8):
    ax = axes[cid // 4, cid % 4]
    ax.imshow(apply_condition(sample_img, cid))
    ax.set_title(condition_names[cid])
    ax.axis("off")

fig.suptitle(f"Synthetic adverse-condition examples | Label: {class_names[sample_label]}",
             fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dataset_examples.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Dataset wrappers

Three dataset classes, one for each training regime.

- **`CleanGTSRBDataset`** — returns `(image, sign_label)` from clean images. Used by Baseline.
- **`RobustGTSRBDataset`** — returns `(image, sign_label)`, applying a random corruption with
  probability `corruption_prob`. Used by Augmented.
- **`MultiTaskGTSRBDataset`** — returns `(image, sign_label, condition_label)`, applying a random
  corruption with probability `corruption_prob` and recording which one. Used by Multi-Task.

`FixedConditionGTSRBDataset` is for evaluation only — applies one fixed corruption to every image.

In [ ]:
class CleanGTSRBDataset(Dataset):
    """Returns clean images. (image, sign_label)."""
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.transform = base, indices, transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.base[self.indices[idx]]
        return self.transform(image), label


class RobustGTSRBDataset(Dataset):
    """Random-corruption dataset for the Augmented model. Returns (image, sign_label)."""
    def __init__(self, base, indices, transform, corruption_prob=0.7):
        self.base, self.indices, self.transform = base, indices, transform
        self.corruption_prob = corruption_prob
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.base[self.indices[idx]]
        if random.random() < self.corruption_prob:
            cid = random.randint(1, 7)
            image = apply_condition(image, cid)
        return self.transform(image), label


class MultiTaskGTSRBDataset(Dataset):
    """Multi-task dataset. Returns (image, sign_label, condition_label)."""
    def __init__(self, base, indices, transform, corruption_prob=0.7):
        self.base, self.indices, self.transform = base, indices, transform
        self.corruption_prob = corruption_prob
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.base[self.indices[idx]]
        if random.random() < self.corruption_prob:
            cid = random.randint(1, 7)
            image = apply_condition(image, cid)
        else:
            cid = 0
        return self.transform(image), label, cid


class FixedConditionGTSRBDataset(Dataset):
    """Applies one fixed condition_id to every image. For evaluation."""
    def __init__(self, base, indices, transform, condition_id):
        self.base, self.indices, self.transform = base, indices, transform
        self.condition_id = condition_id
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        image, label = self.base[self.indices[idx]]
        image = apply_condition(image, self.condition_id)
        return self.transform(image), label, self.condition_id


In [ ]:
# Build the three training loaders
baseline_train_ds  = CleanGTSRBDataset (train_dataset_raw, train_indices, clean_transform)
augmented_train_ds = RobustGTSRBDataset(train_dataset_raw, train_indices, clean_transform, CORRUPTION_PROB)
multitask_train_ds = MultiTaskGTSRBDataset(train_dataset_raw, train_indices, clean_transform, CORRUPTION_PROB)

dl_kwargs = dict(batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
baseline_train_loader  = DataLoader(baseline_train_ds,  **dl_kwargs)
augmented_train_loader = DataLoader(augmented_train_ds, **dl_kwargs)
multitask_train_loader = DataLoader(multitask_train_ds, **dl_kwargs)

# Per-condition test loaders — same across all models
condition_test_loaders = {}
for cid, cname in condition_names.items():
    ds = FixedConditionGTSRBDataset(test_dataset_raw, test_indices, clean_transform, cid)
    condition_test_loaders[cname] = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                               num_workers=2, pin_memory=True)

# Clean test loader (for in-training validation)
clean_val_loader = condition_test_loaders["clean"]

print("All loaders ready.")
print("Train sizes:    baseline =", len(baseline_train_ds),
      "| augmented =", len(augmented_train_ds),
      "| multitask =", len(multitask_train_ds))
print("Per-condition test loaders:", list(condition_test_loaders.keys()))


## 6. Model architectures

**Single-head (Baseline, Augmented):** a stock ResNet-18 with the final FC layer replaced by
a 43-class head.

**Multi-Task (Multi-Task model):** a ResNet-18 backbone whose final FC layer is *removed*,
followed by two parallel heads:

- `sign_head`     — Linear(512 → 43)
- `condition_head` — Linear(512 → 8)

The shared backbone is forced to encode features useful for *both* tasks. Final loss is
$$\mathcal{L} = \mathcal{L}_{\text{sign}} + \lambda \cdot \mathcal{L}_{\text{condition}}$$
with $\lambda$ = `AUX_LOSS_WEIGHT` (default 0.3).

In [ ]:
def create_single_head_model():
    """ResNet-18, ImageNet-pretrained, with a 43-class output head."""
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m


class MultiTaskResNet18(nn.Module):
    """Shared ResNet-18 backbone with two output heads:
        - sign_head     : 43-class traffic sign prediction (primary task)
        - condition_head: 8-class corruption-type prediction (auxiliary task)
    """
    def __init__(self, num_classes=43, num_conditions=8):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        feat_dim = backbone.fc.in_features      # 512
        backbone.fc = nn.Identity()             # remove the original head
        self.backbone = backbone
        self.sign_head      = nn.Linear(feat_dim, num_classes)
        self.condition_head = nn.Linear(feat_dim, num_conditions)

    def forward(self, x):
        feats = self.backbone(x)
        return self.sign_head(feats), self.condition_head(feats)


print("Architectures defined.")
print("MultiTaskResNet18 parameter count:",
      sum(p.numel() for p in MultiTaskResNet18().parameters()))


## 7. Training loops

Two trainers — one for single-head models, one for multi-task. Both save the best checkpoint
(by **clean validation accuracy**) and return the full training history.

In [ ]:
def train_single_head(model, train_loader, val_loader, epochs, lr, save_path):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_acc": []}
    best_acc = 0.0

    for epoch in range(epochs):
        # ---- train ----
        model.train()
        running = 0.0
        for images, labels in tqdm(train_loader, desc=f"[single] epoch {epoch+1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running += loss.item()
        avg_loss = running / len(train_loader)

        # ---- validate (on clean) ----
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for batch in val_loader:
                images, labels = batch[0].to(device), batch[1]
                preds.extend(model(images).argmax(1).cpu().numpy())
                gts.extend(labels.numpy())
        val_acc = accuracy_score(gts, preds)

        history["train_loss"].append(avg_loss)
        history["val_acc"].append(val_acc)
        print(f"  epoch {epoch+1}: loss = {avg_loss:.4f} | val_acc = {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)

    print(f"Best val acc: {best_acc:.4f}  →  {save_path}")
    return model, history


def train_multi_task(model, train_loader, val_loader, epochs, lr, save_path,
                     aux_weight=AUX_LOSS_WEIGHT):
    model = model.to(device)
    criterion_sign = nn.CrossEntropyLoss()
    criterion_cond = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss_total": [], "train_loss_sign": [], "train_loss_cond": [],
               "val_acc_sign": [], "val_acc_cond": []}
    best_acc = 0.0

    for epoch in range(epochs):
        # ---- train ----
        model.train()
        run_total, run_sign, run_cond = 0.0, 0.0, 0.0
        for images, sign_labels, cond_labels in tqdm(train_loader,
                                                     desc=f"[multi]  epoch {epoch+1}/{epochs}"):
            images       = images.to(device)
            sign_labels  = sign_labels.to(device)
            cond_labels  = cond_labels.to(device)

            optimizer.zero_grad()
            sign_logits, cond_logits = model(images)
            loss_sign = criterion_sign(sign_logits, sign_labels)
            loss_cond = criterion_cond(cond_logits, cond_labels)
            loss      = loss_sign + aux_weight * loss_cond
            loss.backward()
            optimizer.step()

            run_total += loss.item()
            run_sign  += loss_sign.item()
            run_cond  += loss_cond.item()

        n = len(train_loader)
        avg_total, avg_sign, avg_cond = run_total/n, run_sign/n, run_cond/n

        # ---- validate ----
        model.eval()
        sp, sg, cp, cg = [], [], [], []
        with torch.no_grad():
            for batch in val_loader:
                images = batch[0].to(device)
                sign_labels = batch[1]
                cond_labels = batch[2] if len(batch) > 2 else None
                sl, cl = model(images)
                sp.extend(sl.argmax(1).cpu().numpy())
                sg.extend(sign_labels.numpy())
                cp.extend(cl.argmax(1).cpu().numpy())
                if cond_labels is not None:
                    if torch.is_tensor(cond_labels):
                        cg.extend(cond_labels.numpy())
                    else:
                        cg.extend(np.array(cond_labels))
        val_sign = accuracy_score(sg, sp)
        val_cond = accuracy_score(cg, cp) if len(cg) == len(cp) and len(cg) > 0 else float("nan")

        history["train_loss_total"].append(avg_total)
        history["train_loss_sign"].append(avg_sign)
        history["train_loss_cond"].append(avg_cond)
        history["val_acc_sign"].append(val_sign)
        history["val_acc_cond"].append(val_cond)
        print(f"  epoch {epoch+1}: total={avg_total:.4f} (sign={avg_sign:.4f}, "
              f"cond={avg_cond:.4f}) | val_sign={val_sign:.4f} val_cond={val_cond:.4f}")

        if val_sign > best_acc:
            best_acc = val_sign
            torch.save(model.state_dict(), save_path)

    print(f"Best val sign acc: {best_acc:.4f}  →  {save_path}")
    return model, history


## 8. Train all three models

In [ ]:
# ---- Model 1: Baseline ----
print("="*60)
print("Training BASELINE (clean only, single-head)")
print("="*60)
baseline_model = create_single_head_model()
baseline_model, baseline_hist = train_single_head(
    baseline_model, baseline_train_loader, clean_val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE,
    save_path=str(MODELS_DIR / "baseline_resnet18.pth"),
)


In [ ]:
# ---- Model 2: Augmented ----
print("="*60)
print("Training AUGMENTED (clean + 7 corruptions, single-head)")
print("="*60)
augmented_model = create_single_head_model()
augmented_model, augmented_hist = train_single_head(
    augmented_model, augmented_train_loader, clean_val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE,
    save_path=str(MODELS_DIR / "augmented_resnet18.pth"),
)


In [ ]:
# ---- Model 3: Multi-Task ----
print("="*60)
print("Training MULTI-TASK (clean + 7 corruptions, dual-head)")
print("="*60)
multitask_model = MultiTaskResNet18(NUM_CLASSES, NUM_CONDITIONS)
multitask_model, multitask_hist = train_multi_task(
    multitask_model, multitask_train_loader, clean_val_loader,
    epochs=EPOCHS, lr=LEARNING_RATE,
    save_path=str(MODELS_DIR / "multitask_resnet18.pth"),
    aux_weight=AUX_LOSS_WEIGHT,
)


In [ ]:
# Plot training curves for all three models
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss
axes[0].plot(baseline_hist["train_loss"],  label="Baseline",  marker="o")
axes[0].plot(augmented_hist["train_loss"], label="Augmented", marker="s")
axes[0].plot(multitask_hist["train_loss_total"], label="Multi-Task (total)", marker="^")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Validation accuracy on clean
axes[1].plot(baseline_hist["val_acc"],     label="Baseline",  marker="o")
axes[1].plot(augmented_hist["val_acc"],    label="Augmented", marker="s")
axes[1].plot(multitask_hist["val_acc_sign"], label="Multi-Task", marker="^")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Validation Accuracy (clean)")
axes[1].set_title("Clean Validation Accuracy"); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_curves.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Per-condition evaluation (sign accuracy)

Each model is evaluated on the same 8 fixed-condition test sets. Accuracies are absolute
(top-1 over the 43-class output).

In [ ]:
def eval_single_head(model, loader):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch[0].to(device); labels = batch[1]
            preds.extend(model(images).argmax(1).cpu().numpy())
            gts.extend(labels.numpy())
    return accuracy_score(gts, preds)


def eval_multi_task(model, loader):
    """Returns (sign_acc, cond_acc) over the loader."""
    model.eval()
    sp, sg, cp, cg = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            images       = batch[0].to(device)
            sign_labels  = batch[1]
            cond_labels  = batch[2] if len(batch) > 2 else None
            sl, cl = model(images)
            sp.extend(sl.argmax(1).cpu().numpy())
            sg.extend(sign_labels.numpy())
            cp.extend(cl.argmax(1).cpu().numpy())
            if cond_labels is not None:
                if torch.is_tensor(cond_labels):
                    cg.extend(cond_labels.numpy())
                else:
                    cg.extend(np.array(cond_labels))
    sign_acc = accuracy_score(sg, sp)
    cond_acc = accuracy_score(cg, cp) if len(cg) == len(cp) and len(cg) > 0 else float("nan")
    return sign_acc, cond_acc


# Per-condition sign accuracy for all three models
results = {"Baseline": {}, "Augmented": {}, "Multi-Task": {}}
mt_cond_acc_per_loader = {}

for cname, loader in condition_test_loaders.items():
    results["Baseline"][cname]   = eval_single_head(baseline_model,  loader)
    results["Augmented"][cname]  = eval_single_head(augmented_model, loader)
    sign_acc, cond_acc           = eval_multi_task(multitask_model,  loader)
    results["Multi-Task"][cname] = sign_acc
    mt_cond_acc_per_loader[cname] = cond_acc
    print(f"{cname:16s}  baseline={results['Baseline'][cname]:.4f}  "
          f"augmented={results['Augmented'][cname]:.4f}  "
          f"multitask={results['Multi-Task'][cname]:.4f}  "
          f"(mt cond_acc={cond_acc:.4f})")


In [ ]:
# Build comparison table
rows = []
for cname in condition_names.values():
    b = results["Baseline"][cname]
    a = results["Augmented"][cname]
    m = results["Multi-Task"][cname]
    rows.append({
        "condition":         cname,
        "Baseline":          round(b, 4),
        "Augmented":         round(a, 4),
        "MultiTask":         round(m, 4),
        "Aug_minus_Base":    round(a - b, 4),
        "MT_minus_Aug":      round(m - a, 4),
        "MT_minus_Base":     round(m - b, 4),
        "MT_cond_acc":       round(mt_cond_acc_per_loader[cname], 4),
    })
metrics_df = pd.DataFrame(rows).set_index("condition")

# Performance drops relative to clean
clean_b = metrics_df.loc["clean", "Baseline"]
clean_a = metrics_df.loc["clean", "Augmented"]
clean_m = metrics_df.loc["clean", "MultiTask"]
metrics_df["Baseline_Drop"]  = (clean_b - metrics_df["Baseline"]).round(4)
metrics_df["Augmented_Drop"] = (clean_a - metrics_df["Augmented"]).round(4)
metrics_df["MultiTask_Drop"] = (clean_m - metrics_df["MultiTask"]).round(4)

metrics_df.to_csv(RESULTS_DIR / "metrics.csv")
metrics_df


## 10. Result visualizations

In [ ]:
# Bar chart: per-condition accuracy across all 3 models
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

conditions = list(metrics_df.index)
x = np.arange(len(conditions))
width = 0.27

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width, metrics_df["Baseline"],  width, label="Baseline",   color="#4C72B0", edgecolor="black")
ax.bar(x,         metrics_df["Augmented"], width, label="Augmented",  color="#DD8452", edgecolor="black")
ax.bar(x + width, metrics_df["MultiTask"], width, label="Multi-Task", color="#55A868", edgecolor="black")

ax.set_ylabel("Accuracy", fontsize=12, fontweight="bold")
ax.set_xlabel("Test Condition", fontsize=12, fontweight="bold")
ax.set_title("Per-Condition Accuracy: Baseline vs Augmented vs Multi-Task",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xticks(x); ax.set_xticklabels(conditions, rotation=30, ha="right")
ax.set_ylim(0, 1.02)
ax.yaxis.grid(True, linestyle="--", alpha=0.6); ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_by_condition.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Performance drop relative to clean
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - width, metrics_df["Baseline_Drop"],  width, label="Baseline",   color="#4C72B0", edgecolor="black")
ax.bar(x,         metrics_df["Augmented_Drop"], width, label="Augmented",  color="#DD8452", edgecolor="black")
ax.bar(x + width, metrics_df["MultiTask_Drop"], width, label="Multi-Task", color="#55A868", edgecolor="black")

ax.set_ylabel("Accuracy Drop vs Clean", fontsize=12, fontweight="bold")
ax.set_xlabel("Test Condition",         fontsize=12, fontweight="bold")
ax.set_title("Performance Drop Under Adverse Conditions (lower = more robust)",
             fontsize=14, fontweight="bold", pad=14)
ax.set_xticks(x); ax.set_xticklabels(conditions, rotation=30, ha="right")
ax.yaxis.grid(True, linestyle="--", alpha=0.6); ax.set_axisbelow(True)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "performance_drop_by_condition.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Confusion matrix on the worst condition (rain) for the multi-task model
def collect_predictions(model, loader, multi_task=False):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch[0].to(device); labels = batch[1]
            if multi_task:
                logits, _ = model(images)
            else:
                logits = model(images)
            preds.extend(logits.argmax(1).cpu().numpy())
            gts.extend(labels.numpy())
    return np.array(gts), np.array(preds)

rain_loader = condition_test_loaders["rain"]
gts_b, preds_b = collect_predictions(baseline_model,  rain_loader)
gts_m, preds_m = collect_predictions(multitask_model, rain_loader, multi_task=True)

cm_b = confusion_matrix(gts_b, preds_b, labels=list(range(NUM_CLASSES)))
cm_m = confusion_matrix(gts_m, preds_m, labels=list(range(NUM_CLASSES)))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, cm, title in zip(axes, [cm_b, cm_m], ["Baseline on Rain", "Multi-Task on Rain"]):
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix_rain.png", dpi=200, bbox_inches="tight")
plt.show()


## 11. Qualitative results

Eight successes and eight failures of the **Multi-Task model on the rain condition**, with both
predicted sign and predicted condition.

In [ ]:
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std =[1/0.229,      1/0.224,      1/0.225])

def tensor_to_image(t):
    img = inv_normalize(t.cpu()).clamp(0, 1).permute(1, 2, 0).numpy()
    return img

def collect_examples(model, loader, max_success=8, max_failure=8, multi_task=True):
    model.eval()
    succ, fail = [], []
    with torch.no_grad():
        for batch in loader:
            images = batch[0]; sign_labels = batch[1]
            cond_labels = batch[2] if len(batch) > 2 else None
            images_d = images.to(device)
            if multi_task:
                sign_logits, cond_logits = model(images_d)
                sign_probs = torch.softmax(sign_logits, dim=1)
                cond_probs = torch.softmax(cond_logits, dim=1)
                s_conf, s_pred = torch.max(sign_probs, dim=1)
                c_conf, c_pred = torch.max(cond_probs, dim=1)
            else:
                logits = model(images_d)
                probs = torch.softmax(logits, dim=1)
                s_conf, s_pred = torch.max(probs, dim=1)
                c_pred = torch.zeros_like(s_pred); c_conf = torch.zeros_like(s_conf)

            for i in range(len(images)):
                ex = {
                    "image":     images[i],
                    "true_sign": int(sign_labels[i]),
                    "pred_sign": int(s_pred[i].cpu()),
                    "sign_conf": float(s_conf[i].cpu()),
                    "pred_cond": int(c_pred[i].cpu()),
                    "cond_conf": float(c_conf[i].cpu()),
                }
                if ex["true_sign"] == ex["pred_sign"] and len(succ) < max_success:
                    succ.append(ex)
                if ex["true_sign"] != ex["pred_sign"] and len(fail) < max_failure:
                    fail.append(ex)
                if len(succ) >= max_success and len(fail) >= max_failure:
                    return succ, fail
    return succ, fail


def plot_examples(examples, title, save_path):
    n = len(examples)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4.5 * rows))
    axes = axes.flatten() if n > 1 else [axes]
    for i, ex in enumerate(examples):
        axes[i].imshow(tensor_to_image(ex["image"]))
        axes[i].axis("off")
        true_name = class_names[ex["true_sign"]][:24]
        pred_name = class_names[ex["pred_sign"]][:24]
        cond_name = condition_names[ex["pred_cond"]]
        axes[i].set_title(
            f"True: {true_name}\nPred: {pred_name} ({ex['sign_conf']:.2f})\n"
            f"Cond: {cond_name} ({ex['cond_conf']:.2f})",
            fontsize=10
        )
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle(title, fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


succ, fail = collect_examples(multitask_model, rain_loader, max_success=8, max_failure=8)
plot_examples(succ, "Multi-Task: Successes on Rain",
              RESULTS_DIR / "qualitative_successes_rain.png")
plot_examples(fail, "Multi-Task: Failures on Rain",
              RESULTS_DIR / "qualitative_failures_rain.png")


## 12. Ablation — corruption probability

We retrain the **Augmented** model with corruption probabilities $p \in \{0.0, 0.3, 0.5, 0.7, 1.0\}$
and measure mean accuracy across the 7 corruption types. This isolates the effect of corruption
exposure rate on robustness.

(We use the augmented single-head model rather than multi-task here for speed; the relative
trend transfers.)

In [ ]:
ablation_probs = [0.0, 0.3, 0.5, 0.7, 1.0]
ablation_results = {}

for p in ablation_probs:
    print(f"\n--- Ablation: corruption_prob = {p} ---")
    ds = RobustGTSRBDataset(train_dataset_raw, train_indices, clean_transform, corruption_prob=p)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    model = create_single_head_model()
    save_path = MODELS_DIR / f"ablation_p{int(p*100):03d}.pth"
    model, _ = train_single_head(model, loader, clean_val_loader,
                                 epochs=EPOCHS, lr=LEARNING_RATE,
                                 save_path=str(save_path))
    # evaluate on each corrupted condition
    accs = {}
    for cname, cloader in condition_test_loaders.items():
        accs[cname] = eval_single_head(model, cloader)
    ablation_results[p] = accs
    corrupted_mean = np.mean([accs[c] for c in condition_names.values() if c != "clean"])
    print(f"  clean acc: {accs['clean']:.4f}   mean corrupted acc: {corrupted_mean:.4f}")

ablation_df = pd.DataFrame(ablation_results).T
ablation_df.index.name = "corruption_prob"
ablation_df["mean_corrupted"] = ablation_df.drop(columns=["clean"]).mean(axis=1)
ablation_df.to_csv(RESULTS_DIR / "ablation_corruption_prob.csv")
ablation_df.round(4)


In [ ]:
# Plot ablation results
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ablation_df.index, ablation_df["clean"],          marker="o", label="Clean accuracy", linewidth=2)
ax.plot(ablation_df.index, ablation_df["mean_corrupted"], marker="s", label="Mean corrupted accuracy", linewidth=2)
ax.set_xlabel("Corruption Probability During Training", fontsize=12, fontweight="bold")
ax.set_ylabel("Test Accuracy",                          fontsize=12, fontweight="bold")
ax.set_title("Ablation: Effect of Corruption Probability on Robustness",
             fontsize=13, fontweight="bold", pad=12)
ax.grid(alpha=0.3); ax.legend(loc="lower right")
ax.set_ylim(0.4, 1.0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ablation_corruption_prob.png", dpi=300, bbox_inches="tight")
plt.show()


## 13. Real-world evaluation

We test the multi-task model on a small hand-curated set of real adverse-condition images
(rain, fog, glare, night). Two passes:

1. **Uncropped images** — qualitative pass. Outputs prediction + condition + confidence.
   Reported as *prediction stability under domain shift*, since these are uncropped scenes
   and not directly comparable to the cropped GTSRB training distribution.
2. **Cropped + labeled set** — quantitative pass. If `real_cropped/` is provided with class
   labels in filenames (e.g. `14_rain_01.jpg` for class 14 = Stop), we compute accuracy.

In [ ]:
# ---- Upload real_adverse_converted.zip if running fresh ----
from google.colab import files
import zipfile

print("Upload real_adverse_converted.zip (uncropped scene images organized by condition)")
uploaded = files.upload()
for fn in uploaded:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("/content")
        print("Extracted:", fn)

REAL_DIR = Path("/content/real_adverse_converted")
print("Real adverse folder exists:", REAL_DIR.exists())


In [ ]:
# ---- Predict on uncropped images, multi-task style ----
def predict_multi(model, img_path, transform):
    img = Image.open(img_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        sl, cl = model(x)
        sp = torch.softmax(sl, dim=1); cp = torch.softmax(cl, dim=1)
        s_conf, s_pred = torch.max(sp, dim=1)
        c_conf, c_pred = torch.max(cp, dim=1)
    return {
        "image":     img,
        "pred_sign": int(s_pred.cpu()), "sign_conf": float(s_conf.cpu()),
        "pred_cond": int(c_pred.cpu()), "cond_conf": float(c_conf.cpu()),
    }

real_records = []
if REAL_DIR.exists():
    for cond_folder in sorted(REAL_DIR.iterdir()):
        if cond_folder.is_dir() and not cond_folder.name.startswith("."):
            for img_path in sorted(cond_folder.glob("*.jpg")):
                r = predict_multi(multitask_model, img_path, clean_transform)
                r["true_condition_folder"] = cond_folder.name
                r["filename"] = img_path.name
                real_records.append(r)
print(f"Collected {len(real_records)} real-world predictions.")


In [ ]:
# Visualize a 4x4 grid of multi-task predictions on real-world images
import math
n = min(16, len(real_records))
if n > 0:
    cols = 4; rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4.2 * rows))
    axes = axes.flatten()
    for i in range(n):
        r = real_records[i]
        axes[i].imshow(r["image"])
        axes[i].axis("off")
        sign_str = class_names[r["pred_sign"]][:22]
        cond_str = condition_names[r["pred_cond"]]
        axes[i].set_title(
            f"folder: {r['true_condition_folder']}\n"
            f"sign: {sign_str} ({r['sign_conf']:.2f})\n"
            f"cond: {cond_str} ({r['cond_conf']:.2f})",
            fontsize=10
        )
    for j in range(n, len(axes)):
        axes[j].axis("off")
    fig.suptitle("Multi-Task Predictions on Real-World Adverse Images",
                 fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "real_world_predictions.png", dpi=180, bbox_inches="tight")
    plt.show()


In [ ]:
# Confidence summary by real-world condition folder
real_summary = defaultdict(lambda: {"sign_confs": [], "cond_confs": [], "cond_preds": []})
for r in real_records:
    f = r["true_condition_folder"]
    real_summary[f]["sign_confs"].append(r["sign_conf"])
    real_summary[f]["cond_confs"].append(r["cond_conf"])
    real_summary[f]["cond_preds"].append(condition_names[r["pred_cond"]])

print("\nReal-world (uncropped) summary:")
for cond, vals in real_summary.items():
    print(f"  {cond:8s}  N={len(vals['sign_confs']):2d}  "
          f"avg sign conf = {np.mean(vals['sign_confs']):.3f}   "
          f"avg cond conf = {np.mean(vals['cond_confs']):.3f}   "
          f"top cond preds = {pd.Series(vals['cond_preds']).value_counts().head(2).to_dict()}")


In [ ]:
# ---- OPTIONAL: cropped + labeled real test set ----
# If you have a cropped folder where each filename starts with the class id,
# e.g.  14_stop_rain_01.jpg, 1_speed30_fog_03.jpg, etc., place it at:
#       /content/real_cropped/{condition}/{class_id}_*.jpg
# This block computes real-world ACCURACY, not just confidence.

REAL_CROPPED = Path("/content/real_cropped")
if REAL_CROPPED.exists():
    cropped_records = []
    for cond_folder in sorted(REAL_CROPPED.iterdir()):
        if cond_folder.is_dir():
            for img_path in sorted(cond_folder.glob("*.jpg")):
                # parse class id from filename prefix
                stem = img_path.stem
                try:
                    cls_id = int(stem.split("_")[0])
                except Exception:
                    print(f"Skipping (can't parse class): {img_path.name}")
                    continue
                r = predict_multi(multitask_model, img_path, clean_transform)
                r["true_class"] = cls_id
                r["filename"] = img_path.name
                r["correct"]  = (r["pred_sign"] == cls_id)
                r["true_condition_folder"] = cond_folder.name
                cropped_records.append(r)

    if cropped_records:
        df = pd.DataFrame(cropped_records)
        overall_acc = df["correct"].mean()
        print(f"\nCropped real-world accuracy: {overall_acc:.4f}  (N={len(df)})")
        print(df.groupby("true_condition_folder")["correct"].agg(["sum", "count", "mean"]).round(3))
        df.drop(columns=["image"]).to_csv(RESULTS_DIR / "real_cropped_results.csv", index=False)
else:
    print("No /content/real_cropped folder found — skipping quantitative real-world step.")
    print("(See the 'Crop & label' instructions in the companion document to enable this.)")


## 14. Package everything for download

In [ ]:
# Bundle results + models into a single zip
import shutil
zip_path = "/content/traffic_sign_robustness_project.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', PROJECT_DIR)
print("Created:", zip_path)

# List outputs
print("\nResults files:")
for p in sorted(RESULTS_DIR.iterdir()):
    print("  ", p.name)
print("\nModel files:")
for p in sorted(MODELS_DIR.iterdir()):
    print("  ", p.name)


In [ ]:
# Download the bundle
from google.colab import files
files.download(zip_path)


---

**Done.** You now have:

- `models/baseline_resnet18.pth` — clean-only single-head
- `models/augmented_resnet18.pth` — clean+corruptions single-head
- `models/multitask_resnet18.pth` — clean+corruptions dual-head (for the demo)
- `models/ablation_p*.pth` — five corruption-probability ablation checkpoints
- `results/metrics.csv` + `ablation_corruption_prob.csv`
- `results/*.png` — all plots and qualitative grids

The **multi-task model** is the one to ship in `demo.zip`.